# 🎯 Outlier Detection and Treatment in Machine Learning

Outliers are extreme observations that deviate significantly from the overall pattern of a dataset. They can distort statistical metrics (mean, variance), violate normality assumptions, and skew model parameters (especially linear and distance-based estimators like Linear Regression and KNN).

This comprehensive guide covers:
- **Visual Detection**: Boxplots and Kernel Density Distributions
- **Empirical Filtering**: Manual thresholding
- **Interquartile Range (IQR) Method**: Tukey's fences for skewed data
- **Outlier Handling Techniques**: Trimming (removal) vs. Capping (Winsorization)
- **Z-Score Method**: Standard score thresholding for normal distributions
- **Comparative Diagnostic Visualizations**: Pre- vs. Post-treatment evaluation


### 1. Library Imports
Import `seaborn`, `pandas`, `numpy`, `matplotlib.pyplot`, and `scipy.stats` for statistical modeling and visualization.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

import warnings
warnings.filterwarnings('ignore')


### 2. Loading the Dataset
Load the benchmark `tips` dataset from Seaborn and inspect the first records and dimensions.


In [ ]:
df = sns.load_dataset('tips')
print(f"Dataset Shape: {df.shape}")
df.head()


### 3. Visualizing Distribution with Boxplot
Generate a vertical boxplot for `total_bill` to observe the median, interquartile spread, and individual data points plotted beyond the whiskers as potential outliers.


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(y=df['total_bill'], color='skyblue')
plt.title('Boxplot of Total Bill (Detecting Upper-Tail Outliers)')
plt.ylabel('Total Bill ($)')
plt.show()

print("Descriptive Statistics for Total Bill:")
print(df['total_bill'].describe())


### 4. Empirical Filtering: Counting Extreme Bills
Count how many restaurant parties incurred a `total_bill` greater than $40 to assess the upper-tail frequency.


In [ ]:
high_bill_count = (df['total_bill'] > 40).sum()
print(f"Number of bills > $40: {high_bill_count} ({high_bill_count / len(df) * 100:.2f}% of total records)")
df[df['total_bill'] > 40].head()


### 5. Manual Threshold Filtering & Index Reset
Demonstrate manual row deletion based on the hard threshold (`total_bill > 40`), followed by resetting the DataFrame index to maintain contiguous row indices.


In [ ]:
df_manual = df.copy()

# Drop rows where total_bill > 40
df_manual = df_manual[df_manual['total_bill'] <= 40].reset_index(drop=True)

print(f"Original shape: {df.shape}, Manual filtered shape: {df_manual.shape}")
df_manual.head()


### 6. Interquartile Range (IQR) Method: Theory & Math
The Interquartile Range (IQR) method (Tukey's fences) is a robust, non-parametric approach that does not assume a Gaussian distribution:
- **First Quartile ($Q_1$)**: 25th percentile
- **Third Quartile ($Q_3$)**: 75th percentile
- **Interquartile Range ($IQR$)**: $IQR = Q_3 - Q_1$
- **Lower Fence**: $Lower = Q_1 - 1.5 \times IQR$
- **Upper Fence**: $Upper = Q_3 + 1.5 \times IQR$

Any data point falling outside $[Lower, Upper]$ is considered an outlier.


In [ ]:
Q1 = df['total_bill'].quantile(0.25)
Q3 = df['total_bill'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Q1 (25th percentile): ${Q1:.2f}")
print(f"Q3 (75th percentile): ${Q3:.2f}")
print(f"IQR: ${IQR:.2f}")
print(f"Lower Fence: ${lower_fence:.2f} (values below this are lower outliers)")
print(f"Upper Fence: ${upper_fence:.2f} (values above this are upper outliers)")


### 7. Identifying IQR Outliers
Filter and display the records identified as statistical outliers under the 1.5x IQR rule.


In [ ]:
iqr_outliers = df[(df['total_bill'] < lower_fence) | (df['total_bill'] > upper_fence)]
print(f"Total IQR Outliers Detected: {len(iqr_outliers)} ({len(iqr_outliers) / len(df) * 100:.2f}%)")
iqr_outliers.sort_values(by='total_bill', ascending=False)


### 8. Treatment Technique 1: Trimming (Data Removal)
Trimming removes all rows exceeding the upper or lower fences.
- **Advantage**: Completely purges anomalous observations.
- **Disadvantage**: Loss of sample size and potential removal of legitimate high-value customers.


In [ ]:
df_trimmed = df[(df['total_bill'] >= lower_fence) & (df['total_bill'] <= upper_fence)].copy()
df_trimmed.reset_index(drop=True, inplace=True)

print(f"Original Count: {len(df)}, Trimmed Count: {len(df_trimmed)}")
print(f"Rows Removed: {len(df) - len(df_trimmed)}")


### 9. Treatment Technique 2: Capping / Winsorization
Capping (Winsorization) clamps outlier values to the calculated boundaries ($Lower$ and $Upper$) instead of dropping rows.
- **Advantage**: Retains 100% of sample observations while neutralizing the destabilizing leverage of extreme values on linear estimators.


In [ ]:
df_capped = df.copy()

# Clamp total_bill values to [lower_fence, upper_fence]
df_capped['total_bill'] = np.clip(df_capped['total_bill'], lower_fence, upper_fence)

print(f"Capped Total Bill Min: ${df_capped['total_bill'].min():.2f}")
print(f"Capped Total Bill Max: ${df_capped['total_bill'].max():.2f}")
print(f"Retained rows: {len(df_capped)}")


### 10. Statistical Outlier Detection: Z-Score Method
The Z-Score measures how many standard deviations an observation lies from the sample mean:
$$Z = \frac{X - \mu}{\sigma}$$
Under the empirical 3-sigma rule for approximately normal distributions, observations with $|Z| > 3$ represent extreme anomalies (less than 0.3% probability).


In [ ]:
mean_val = df['total_bill'].mean()
std_val = df['total_bill'].std()

df['z_score'] = (df['total_bill'] - mean_val) / std_val

z_outliers = df[np.abs(df['z_score']) > 3]
print(f"Mean: ${mean_val:.2f}, Std Dev: ${std_val:.2f}")
print(f"Outliers with |Z| > 3: {len(z_outliers)}")
z_outliers[['total_bill', 'tip', 'sex', 'z_score']]


### 11. Diagnostic Visualizations: Original vs. Trimmed vs. Capped
Compare the distributions and boxplots of the original, trimmed, and capped datasets side-by-side to visually inspect how outlier treatment normalizes data spread.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Boxplots
sns.boxplot(y=df['total_bill'], ax=axes[0, 0], color='lightcoral')
axes[0, 0].set_title('Original Data (With Outliers)')

sns.boxplot(y=df_trimmed['total_bill'], ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Trimmed Data (Outliers Removed)')

sns.boxplot(y=df_capped['total_bill'], ax=axes[0, 2], color='lightblue')
axes[0, 2].set_title('Capped Data (Winsorized)')

# KDE Distributions
sns.histplot(df['total_bill'], kde=True, ax=axes[1, 0], color='darkred')
axes[1, 0].set_title('Original Distribution (Right-Skewed)')

sns.histplot(df_trimmed['total_bill'], kde=True, ax=axes[1, 1], color='darkgreen')
axes[1, 1].set_title('Trimmed Distribution')

sns.histplot(df_capped['total_bill'], kde=True, ax=axes[1, 2], color='darkblue')
axes[1, 2].set_title('Capped Distribution')

plt.tight_layout()
plt.show()


### 12. Strategy Guide: Choosing the Right Outlier Strategy
| Strategy | Method | Best When | Trade-off |
| :--- | :--- | :--- | :--- |
| **Keep Outliers** | No transformation | Using Tree-based models (Random Forest, XGBoost) | None for robust models |
| **Trimming** | Remove rows | Outliers stem from data entry error or corruption | Reduces sample size |
| **Capping (Winsorization)** | Clamp to $Q_3 + 1.5\times IQR$ | Using Linear Regression, PCA, or KNN | Modifies real values at tails |
| **Log / Power Transform** | $\log(1 + x)$ | Extreme continuous positive skewness (e.g. Income, Prices) | Changes interpretability scale |
